# Data

In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt -n 3

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [4]:
import collections
import numpy as np
import tqdm

def load(limit, path = "stand/log.local.txt"):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    with open(path) as f:
        req = list()
        reqid = None
        models = list()
        prevreqmodel = None
        reqmodel = dict()
        prevmodelid = -1
        bannermodelid = -1
        for i, line in tqdm.tqdm_notebook(enumerate(f)):
            if line.startswith("Model = 6;"):
                prevreqmodel = reqmodel
                reqmodel = dict()

            if line.startswith("Model = "):
                spl = line.split(" ")
                prevmodelid = int(spl[2][:-1])
                bannermodelid = max(bannermodelid , prevmodelid)
                reqmodel[prevmodelid] = readvector(spl[3])
            elif line.startswith("dbid"):
                spl = line.split(" ")
                dbid = int(spl[1][:-1])
                docembs[bannermodelid][dbid] = readvector(spl[2])
            elif line.startswith("seed"):
                if len(requests) >= limit:
                    break
                if req:
                    requests.append((reqid, prevreqmodel, sorted(req)))
                    req = list()
                reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
            else:
                req.append(
                    (int(line.split()[0]), float(line.split()[1]))
                )

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7):
            self.games_count = games_count
            self.key_games = np.random.choice(np.arange(games_count), key_size, replace=False)

            self.requests = requests
            np.random.shuffle(self.requests)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext()

In [5]:
ctx = load(1500)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
101 937 446


# Models

In [6]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in self.ctx.get_requests(t)
        ])
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):
        mses = list()
        results = list()
        for rec, tru in zip(self.recommend(t), self.get_user_scores(t)):
            mses.append((rec - tru) ** 2)
            rec = np.argsort(-rec)
            tru = np.argsort(-tru)
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            results.append(ev(rec[:topsize], tru[:topsize]) / float(topsize))
        print(np.mean(results), np.array(mses).mean(), len(results))
        return np.mean(results)

In [24]:
p = Popular(ctx)
p.fit()
p.get_score("train"), p.get_score("test")

0.4597118463180363 1.5713968168959522 937
0.47199551569506726 1.5107071184973542 446


(0.4597118463180363, 0.47199551569506726)

In [8]:
import tensorflow as tf
import copy

DEFAULT_FIT_KWARGS = {
        "learning_rate": 0.001,
        "n": 500,
        "c": 5000,
        "train_popbias": False,
        "verbose": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        self.fit_kwargs = fit_kwargs
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[2][i][1] for i in ctx.key_games])
                    for r_i in ctx.get_requests(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[2][g_i][1] for r in ctx.get_requests("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in ctx.get_requests("key")
        ])
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return self.embed_users[t]
    
    def get_game_embs(self):
        return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.train_popbias = p["train_popbias"]

        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        loss = tf.keras.losses.MeanSquaredError()
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        for i in range(p["n"]):
            def loss():
                logits = train_user_embs @ self.W @ game_embs.T

                if self.train_popbias:
                    logits += self.pb * self.game_avg_scores["train"]

                v = (
                    tf.reduce_mean((train_user_scores - logits) ** 2)
                    + tf.reduce_mean(self.W * self.W) * p["c"]
                )
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            opt.minimize(loss, [self.W, self.pb] if self.train_popbias else [self.W]).numpy()

    def recommend(self, t):
        logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):

In [9]:
ck = CBKnnV0(ctx)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)


In [10]:
ck.fit(c = 1000)

self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.1529460e-03,  5.2342011e-04, -7.8516325e-04, ...,
        -1.5997984e-03, -1.5134126e-03, -1.5742764e-04],
       [-1.4372021e-05, -3.8387001e-04,  4.6069978e-04, ...,
         7.1960507e-04, -2.4959861e-04, -6.6534430e-04],
       [ 1.1174801e-03,  4.1868509e-04,  2.4483277e-04, ...,
        -5.4665765e-04, -2.5923192e-04, -1.0980505e-03],
       ...,
       [-1.2043410e-03, -5.5185863e-04, -1.6802391e-03, ...,
        -9.2313119e-04, -1.5336662e-03,  1.5021100e-03],
       [ 4.7731682e-04,  1.1759523e-03, -3.4092902e-04, ...,
         1.2597207e-03,  8.4432989e-04, -1.1484474e-05],
       [ 9.8430482e-04, -1.4062297e-03, -4.8073410e-04, ...,
         4.8491597e-04,  1.8779636e-05,  3.5736070e-04]], dtype=float32)>
0-loss =  tf.Tensor(1.6263173771745756, shape=(), dtype=float64)
1.6283077
1.5936562
1.5956352
1.5942916
1.5857915
1.5789894
1.5772946
1.5774539
1.5756809
1.5716325
1.5676576
1.5660086
1.5

In [11]:
ck.get_score("test")

0.5604035874439462 1.9514738 446


0.5604035874439462

In [12]:
ck = CBKnnV0(ctx)
ck.fit(c = 0.1, train_popbias=True)

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-0.00172096, -0.00022817,  0.00023611, ..., -0.00077109,
         0.00163899,  0.00040979],
       [-0.00055901, -0.00143032, -0.00160431, ...,  0.00105531,
         0.00021994, -0.00161132],
       [-0.0001584 ,  0.00111094, -0.00140309, ...,  0.00075169,
         0.00029347,  0.00150251],
       ...,
       [ 0.00029818,  0.00102135, -0.00154344, ..., -0.00094633,
        -0.00133519, -0.00057338],
       [ 0.00133555, -0.00144407, -0.00052845, ..., -0.0014869 ,
         0.00108294,  0.00163233],
       [-0.00078133, -0.00172682, -0.00072445, ..., -0.000155  ,
         0.00135153, -0.00163328]], dtype=float32)>
0-loss =  tf.Tensor(1.6263173771745756, shape=(), dtyp

In [13]:
ck.get_score("test")

0.6006053811659193 2.114506 446


0.6006053811659193

In [14]:
DEFAULT_FIT_KWARGS

{'learning_rate': 0.001,
 'n': 500,
 'c': 5000,
 'train_popbias': False,
 'verbose': True}

In [15]:
ck = CBKnnV0(ctx, fit_kwargs={"c": 0.1, "train_popbias": True})
ck.fit(verbose = False)
ck.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.6006726457399103 2.1178312 446


0.6006726457399103

In [16]:
DEFAULT_FIT_KWARGS

{'learning_rate': 0.001,
 'n': 500,
 'c': 5000,
 'train_popbias': False,
 'verbose': True}

In [17]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games

In [18]:
d = DssmKnn(ctx, 6, fit_kwargs={"c": 0.1, "train_popbias": True})
d.fit(verbose=False)
d.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5345067264573992 1.4824542 446


0.5345067264573992

In [19]:
d = DssmKnn(ctx, 6)
d.fit(n = 1, c = 1, train_popbias=True, verbose=False)
d.get_score("test")

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.4719955156950673 1.5101099 446


0.4719955156950673

In [20]:
for i in range(10):
    d.W = np.eye(*d.W.shape) * i
    d.pb = 1.
    d.get_score("train")

0.4597118463180363 1.5713968168959522 937
0.4933404482390608 1.5962600225273103 937
0.5305656350053362 1.7093720371191974 937
0.5623585912486659 1.9107328606716159 937
0.5855282817502668 2.200342493184571 937
0.5913447171824974 2.5782009346580548 937
0.5840981856990395 3.0443081850920692 937
0.5705656350053362 3.5986642444866166 937
0.5535005336179295 4.241269112841703 937
0.5346211312700107 4.972122790157311 937


In [21]:
for i in range(10):
    d.W = np.eye(*d.W.shape) * i
    d.pb = 1.
    d.get_score("test")

0.47199551569506726 1.5107071184973542 446
0.5044170403587444 1.5363752233478214 446
0.5385650224215246 1.6520792013816132 446
0.5693497757847534 1.8578190525987255 446
0.5902466367713004 2.153594776999164 446
0.5962780269058295 2.53940637458292 446
0.5885426008968611 3.0152538453500077 446
0.5748206278026906 3.581137189300417 446
0.557152466367713 4.237056406434147 446
0.5389686098654708 4.983011496751204 446


# нужны ранжирующие лоссы!

In [27]:
score = dict()
for c in [0, .1, 1, 10, 100, 1000]:
    for m in [6,7,8,9]:
        d = DssmKnn(ctx, m)
        d.fit(c = c, train_popbias=True, verbose=False)
        score_train_ = d.get_score("train")
        score_test_ = d.get_score("test")
        print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
        score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5286125933831377 1.5397363 937
0.5377578475336323 1.4808576 446
c, m =  0 6  score_train_ =  0.5286125933831377  score_test_ =  0.5377578475336323
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5227641408751335 1.5394075 937
0.5330717488789237 1.481133 446
c, m =  0 7  score_train_ =  0.5227641408751335  score_test_ =  0.5330717488789237
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_use

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.45973319103521876 1.5710654 937
0.47204035874439454 1.5103939 446
c, m =  1000 7  score_train_ =  0.45973319103521876  score_test_ =  0.47204035874439454
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.4611632870864461 1.5708507 937
0.4731390134529148 1.5101732 446
c, m =  1000 8  score_train_ =  0.4611632870864461  score_test_ =  0.4731390134529148
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)


In [28]:
score

{(0, 6): 0.5377578475336323,
 (0, 7): 0.5330717488789237,
 (0, 8): 0.5355381165919283,
 (0, 9): 0.48950672645739907,
 (0.1, 6): 0.5342825112107624,
 (0.1, 7): 0.5288789237668161,
 (0.1, 8): 0.5335874439461883,
 (0.1, 9): 0.4916143497757847,
 (1, 6): 0.5181838565022422,
 (1, 7): 0.509439461883408,
 (1, 8): 0.5195964125560538,
 (1, 9): 0.4912780269058296,
 (10, 6): 0.49024663677130037,
 (10, 7): 0.48213004484304933,
 (10, 8): 0.4872645739910314,
 (10, 9): 0.47757847533632286,
 (100, 6): 0.4734977578475337,
 (100, 7): 0.47298206278026905,
 (100, 8): 0.47390134529147987,
 (100, 9): 0.4730044843049327,
 (1000, 6): 0.47199551569506726,
 (1000, 7): 0.47204035874439454,
 (1000, 8): 0.4731390134529148,
 (1000, 9): 0.47199551569506726}

In [26]:
for c in [0, .1, 1, 10, 100, 1000]:
    d = CBKnnV0(ctx)
    d.fit(c = c, train_popbias=True, verbose=False)
    score_train_ = d.get_score("train")
    score_test_ = d.get_score("test")
    print("c, m = ", c, m, " score_train_ = ", score_train_, " score_test_ = ", score_test_)
    score[(c, m)] = score_test_

self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5949839914621131 1.4684364 937
0.6007847533632287 2.1153526 446
c, m =  0 8  score_train_ =  0.5949839914621131  score_test_ =  0.6007847533632287
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (937, 101)
0.5953255069370331 1.4684305 937
0.6005829596412556 2.1192644 446
c, m =  0.1 8  score_train_ =  0.5953255069370331  score_test_ =  0.6005829596412556
self.embed_users['train'].shape =  (937, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_

# EOLN

In [127]:
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from sklearn.metrics import pairwise 

In [ ]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

In [23]:
import matplotlib.pyplot as plt
plt.hist(embed_games.flatten())

(array([1.667877e+06, 2.000000e+01, 6.000000e+00, 4.000000e+00,
        1.000000e+00, 4.000000e+00, 0.000000e+00, 0.000000e+00,
        1.000000e+00, 1.000000e+00]),
 array([9.74706709e-05, 6.56539696e+01, 1.31307842e+02, 1.96961714e+02,
        2.62615586e+02, 3.28269458e+02, 3.93923330e+02, 4.59577202e+02,
        5.25231074e+02, 5.90884946e+02, 6.56538818e+02]),
 <a list of 10 Patch objects>)

In [25]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.005857787810383747 0.4473702031602709
d -> 0.4814559819413093 0.006038374717832957 0.4473702031602709
c -> 0.01746049661399549 0.005959367945823928 0.4473702031602709
w -> 0.4944808126410835 0.0060835214446952595 0.4473702031602709


In [26]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.003182844243792325 0.3336343115124153
d -> 0.36880361173814896 0.0031828442437923255 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.38158013544018066 0.003069977426636569 0.3336343115124153


Тюним

In [135]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 1000
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)
1.3766173
1.2021382
1.1933919
1.1919646
1.1917517
1.1917268
1.1917244
1.1917244
1.1917244
1.1917243


In [28]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.002618510158013544 0.3336343115124153
d -> 0.36880361173814896 0.0029119638826185104 0.3336343115124153
c -> 0.008577878103837472 0.002957110609480813 0.3336343115124153
w -> 0.40388261851015805 0.0027539503386004517 0.3336343115124153


In [29]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.006422121896162527 0.4473702031602709
d -> 0.4814559819413093 0.0060609480812641075 0.4473702031602709
c -> 0.01746049661399549 0.005711060948081265 0.4473702031602709
w -> 0.512020316027088 0.005970654627539504 0.4473702031602709


# dssm?

In [136]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-docblocks[i] @ requests_i[1][i] for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.15483069977426636 0.0031602708803611735 0.3336343115124153
7 -> 0.11663656884875846 0.0034085778781038373 0.3336343115124153
8 -> 0.15386004514672688 0.0032054176072234763 0.3336343115124153
9 -> 0.0362979683972912 0.0032054176072234763 0.3336343115124153
e -> 0.3762076749435666 0.0031602708803611735 0.3336343115124153
d -> 0.36880361173814896 0.0028442437923250565 0.3336343115124153
c -> 0.008577878103837472 0.0029119638826185104 0.3336343115124153
w -> 0.40388261851015805 0.002618510158013544 0.3336343115124153


In [137]:
games_top

array([0.0047713 , 0.00415357, 0.00380251, ..., 0.1423562 , 0.04750573,
       0.07614477])

In [141]:
dssmid = 7
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
7 -> 0.3336343115124153 0.0031602708803611735 0.3336343115124153
x =  1
7 -> 0.35379232505643343 0.0027765237020316025 0.3336343115124153
x =  2
7 -> 0.37090293453724604 0.0031602708803611735 0.3336343115124153
x =  3
7 -> 0.38564334085778784 0.002889390519187359 0.3336343115124153
x =  4
7 -> 0.40259593679458244 0.002663656884875847 0.3336343115124153
x =  5
7 -> 0.4182844243792325 0.002641083521444696 0.3336343115124153
x =  6
7 -> 0.4323702031602709 0.0031602708803611743 0.3336343115124153
x =  7
7 -> 0.4396839729119639 0.0028668171557562076 0.3336343115124153
x =  8
7 -> 0.44162528216704294 0.0034085778781038373 0.3336343115124153
x =  9
7 -> 0.4411286681715576 0.0035665914221218965 0.3336343115124153
x =  10
7 -> 0.4392099322799097 0.0030248306997742664 0.3336343115124153
x =  11
7 -> 0.43462753950338606 0.0030248306997742664 0.3336343115124153
x =  12
7 -> 0.4294356659142212 0.003386004514672686 0.3336343115124153
x =  13
7 -> 0.42151241534988715 0.003363431151241535 0.333

In [142]:
dssmid = 6
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.3336343115124153 0.0030925507900677203 0.3336343115124153
x =  1
6 -> 0.37266365688487585 0.003002257336343115 0.3336343115124153
x =  2
6 -> 0.3972911963882618 0.003340857787810384 0.3336343115124153
x =  3
6 -> 0.42458239277652365 0.0024830699774266367 0.3336343115124153
x =  4
6 -> 0.45449209932279916 0.003115124153498871 0.3336343115124153
x =  5
6 -> 0.4766591422121897 0.0030699774266365687 0.3336343115124153
x =  6
6 -> 0.4895033860045147 0.0032054176072234763 0.3336343115124153
x =  7
6 -> 0.4941083521444696 0.0032731376975169302 0.3336343115124153
x =  8
6 -> 0.49146726862302487 0.003002257336343115 0.3336343115124153
x =  9
6 -> 0.48632054176072237 0.002595936794582393 0.3336343115124153
x =  10
6 -> 0.4764559819413092 0.0028893905191873593 0.3336343115124153
x =  11
6 -> 0.4654853273137698 0.0027313769751693 0.3336343115124153
x =  12
6 -> 0.45309255079006777 0.0030248306997742672 0.3336343115124153
x =  13
6 -> 0.43932279909706545 0.003024830699774266 0.3336343

In [152]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.3336343115124153 0.0029345372460496616 0.3336343115124153
x =  1
9 -> 0.34738148984198647 0.002844243792325057 0.3336343115124153
x =  2
9 -> 0.3593227990970655 0.003002257336343115 0.3336343115124153
x =  3
9 -> 0.3699097065462754 0.0030248306997742672 0.3336343115124153
x =  4
9 -> 0.37024830699774264 0.003024830699774266 0.3336343115124153
x =  5
9 -> 0.3647178329571106 0.002595936794582393 0.3336343115124153
x =  6
9 -> 0.3520767494356659 0.0028668171557562076 0.3336343115124153
x =  7
9 -> 0.3291196388261851 0.003047404063205418 0.3336343115124153
x =  8
9 -> 0.30002257336343113 0.002641083521444696 0.3336343115124153
x =  9
9 -> 0.2689390519187359 0.002889390519187359 0.3336343115124153
x =  10
9 -> 0.24376975169300227 0.0027313769751693 0.3336343115124153
x =  11
9 -> 0.22112866817155757 0.0031828442437923255 0.3336343115124153
x =  12
9 -> 0.20067720090293456 0.0034085778781038373 0.3336343115124153
x =  13
9 -> 0.18354401805869075 0.0030925507900677203 0.33363431

In [143]:
for x in range(20):
    print("x = ", x)
    rank_w = np.argsort(-x*embed_users @ W @ embed_games.T-games_top, axis=1)


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_w, "w"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
w -> 0.3336343115124153 0.0032279909706546274 0.3336343115124153
x =  1
w -> 0.39647855530474035 0.0033182844243792326 0.3336343115124153
x =  2
w -> 0.40907449209932284 0.0030925507900677203 0.3336343115124153
x =  3
w -> 0.4110835214446953 0.0031828442437923255 0.3336343115124153
x =  4
w -> 0.41049661399548537 0.0035214446952595937 0.3336343115124153
x =  5
w -> 0.4100451467268624 0.0034085778781038373 0.3336343115124153
x =  6
w -> 0.4096839729119639 0.003340857787810384 0.3336343115124153
x =  7
w -> 0.40936794582392777 0.0034762979683972913 0.3336343115124153
x =  8
w -> 0.40880361173814905 0.0031602708803611735 0.3336343115124153
x =  9
w -> 0.4087810383747179 0.0030248306997742664 0.3336343115124153
x =  10
w -> 0.4089164785553048 0.002934537246049662 0.3336343115124153
x =  11
w -> 0.4089164785553048 0.0027313769751693 0.3336343115124153
x =  12
w -> 0.40887133182844243 0.003295711060948081 0.3336343115124153
x =  13
w -> 0.4086004514672686 0.002595936794582393 0.333634

In [147]:
print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)


In [148]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T-games_top) ** 2)
            + tf.reduce_mean(W * W) * 150
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
1.3189981
1.1485099
1.1218199
1.110669
1.1049802
1.1020124
1.1004989
1.0997444
1.0993747
1.0991968


In [149]:
rank_w = np.argsort(-embed_users @ W @ embed_games.T-games_top, axis=1)


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_w, "w"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids

            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

w -> 0.4295936794582393 0.002595936794582393 0.3336343115124153


In [41]:
500 -> .4108
250 -> .4117
150 -> .4143
100 -> .4151
050 -> .4142
010 -> .4115
000 -> .4033

SyntaxError: invalid syntax (<ipython-input-41-1b4e77c0b5a7>, line 1)

In [150]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.4941083521444696 0.0028216704288939053 0.3336343115124153
7 -> 0.4396839729119639 0.0032054176072234763 0.3336343115124153
8 -> 0.47376975169300223 0.0033182844243792326 0.3336343115124153
9 -> 0.3291196388261851 0.0028668171557562076 0.3336343115124153
e -> 0.3762076749435666 0.0031376975169300227 0.3336343115124153
d -> 0.36880361173814896 0.0027539503386004517 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.4295936794582393 0.0032054176072234763 0.3336343115124153


In [153]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5568735891647856 0.006151241534988712 0.4473702031602709
7 -> 0.5192212189616252 0.005846501128668171 0.4473702031602709
8 -> 0.567178329571106 0.006455981941309256 0.4473702031602709
9 -> 0.37682844243792324 0.006264108352144469 0.4473702031602709
e -> 0.4731151241534989 0.006218961625282167 0.4473702031602709
d -> 0.4814559819413093 0.006060948081264108 0.4473702031602709
c -> 0.01746049661399549 0.005812641083521445 0.4473702031602709
w -> 0.527155756207675 0.006060948081264108 0.4473702031602709


In [154]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-5 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5759367945823928 0.005711060948081263 0.4473702031602709
7 -> 0.5242776523702032 0.006015801354401806 0.4473702031602709
8 -> 0.5645372460496614 0.005744920993227991 0.4473702031602709
9 -> 0.44339729119638827 0.006015801354401806 0.4473702031602709
e -> 0.4731151241534989 0.005970654627539504 0.4473702031602709
d -> 0.4814559819413093 0.006049661399548532 0.4473702031602709
c -> 0.01746049661399549 0.006309255079006772 0.4473702031602709
w -> 0.527155756207675 0.006230248306997742 0.4473702031602709


In [160]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 200
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.666410835214447 0.01167607223476298 0.5744469525959367
7 -> 0.6234255079006772 0.011834085778781037 0.5744469525959367
8 -> 0.6523137697516931 0.012178329571106095 0.5744469525959367
9 -> 0.5699266365688487 0.011811512415349886 0.5744469525959367
e -> 0.5722686230248306 0.01195823927765237 0.5744469525959367
d -> 0.5961343115124152 0.012076749435665914 0.5744469525959367
c -> 0.030299097065462757 0.012753950338600452 0.5744469525959367
w -> 0.626089164785553 0.011726862302483071 0.5744469525959367


In [161]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 250
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.6831467268623025 0.014785553047404065 0.5979503386004515
7 -> 0.6498690744920992 0.015480812641083523 0.5979503386004515
8 -> 0.6795124153498872 0.01450112866817156 0.5979503386004515
9 -> 0.5858916478555304 0.014961625282167042 0.5979503386004515
e -> 0.592334085778781 0.015431151241534989 0.5979503386004515
d -> 0.6212821670428893 0.015327313769751695 0.5979503386004515
c -> 0.03572460496613995 0.015449209932279913 0.5979503386004515
w -> 0.6479593679458239 0.015462753950338602 0.5979503386004515


In [51]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.5744469525959367 0.012121896162528218 0.5744469525959367
x =  1
6 -> 0.6088205417607224 0.01252257336343115 0.5744469525959367
x =  2
6 -> 0.6479627539503386 0.01204288939051919 0.5744469525959367
x =  3
6 -> 0.664283295711061 0.012088036117381492 0.5744469525959367
x =  4
6 -> 0.6479740406320542 0.011580135440180587 0.5744469525959367
x =  5
6 -> 0.6167832957110609 0.011839729119638827 0.5744469525959367
x =  6
6 -> 0.5822968397291196 0.012025959367945822 0.5744469525959367
x =  7
6 -> 0.5485609480812641 0.012291196388261852 0.5744469525959367
x =  8
6 -> 0.5183069977426636 0.012127539503386006 0.5744469525959367
x =  9
6 -> 0.49157449209932275 0.011930022573363432 0.5744469525959367
x =  10
6 -> 0.46870203160270885 0.012020316027088036 0.5744469525959367
x =  11
6 -> 0.44742099322799095 0.012477426636568848 0.5744469525959367
x =  12
6 -> 0.4283465011286682 0.012116252821670429 0.5744469525959367
x =  13
6 -> 0.41200338600451464 0.012003386004514673 0.5744469525959367
x

In [52]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.4473702031602709 0.0059480812641083515 0.4473702031602709
x =  1
6 -> 0.4830135440180587 0.006241534988713319 0.4473702031602709
x =  2
6 -> 0.5132167042889391 0.006478555304740407 0.4473702031602709
x =  3
6 -> 0.5453160270880361 0.006038374717832957 0.4473702031602709
x =  4
6 -> 0.56813769751693 0.006207674943566591 0.4473702031602709
x =  5
6 -> 0.5727878103837472 0.006015801354401806 0.4473702031602709
x =  6
6 -> 0.5652934537246049 0.005857787810383747 0.4473702031602709
x =  7
6 -> 0.5506546275395033 0.005575620767494356 0.4473702031602709
x =  8
6 -> 0.5308352144469526 0.005744920993227991 0.4473702031602709
x =  9
6 -> 0.5122234762979685 0.006106094808126411 0.4473702031602709
x =  10
6 -> 0.4936230248306998 0.006591422121896162 0.4473702031602709
x =  11
6 -> 0.47469525959367953 0.00618510158013544 0.4473702031602709
x =  12
6 -> 0.45573363431151237 0.006218961625282167 0.4473702031602709
x =  13
6 -> 0.43795711060948084 0.0059480812641083515 0.4473702031602709


In [158]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.4473702031602709 0.0063318284424379225 0.4473702031602709
x =  1
9 -> 0.45654627539503384 0.005846501128668171 0.4473702031602709
x =  2
9 -> 0.4679345372460497 0.006331828442437923 0.4473702031602709
x =  3
9 -> 0.4737810383747178 0.0060270880361173815 0.4473702031602709
x =  4
9 -> 0.46619638826185095 0.006218961625282167 0.4473702031602709
x =  5
9 -> 0.44339729119638827 0.006399548532731377 0.4473702031602709
x =  6
9 -> 0.4103386004514672 0.006399548532731377 0.4473702031602709
x =  7
9 -> 0.37682844243792324 0.005688487584650113 0.4473702031602709
x =  8
9 -> 0.3405079006772009 0.005654627539503386 0.4473702031602709
x =  9
9 -> 0.30753950338600455 0.0060270880361173815 0.4473702031602709
x =  10
9 -> 0.27742663656884875 0.006309255079006772 0.4473702031602709
x =  11
9 -> 0.25196388261851016 0.006162528216704289 0.4473702031602709
x =  12
9 -> 0.23016930022573365 0.005948081264108352 0.4473702031602709
x =  13
9 -> 0.2112641083521445 0.006252821670428894 0.44737020

In [159]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.5744469525959367 0.012392776523702033 0.5744469525959367
x =  1
9 -> 0.584018058690745 0.012020316027088036 0.5744469525959367
x =  2
9 -> 0.586410835214447 0.011653498871331828 0.5744469525959367
x =  3
9 -> 0.5699266365688487 0.012279909706546277 0.5744469525959367
x =  4
9 -> 0.5346388261851016 0.011822799097065465 0.5744469525959367
x =  5
9 -> 0.49005079006772007 0.012133182844243792 0.5744469525959367
x =  6
9 -> 0.44170993227990973 0.011918735891647854 0.5744469525959367
x =  7
9 -> 0.39646726862302484 0.012466139954853276 0.5744469525959367
x =  8
9 -> 0.35639954853273137 0.011952595936794583 0.5744469525959367
x =  9
9 -> 0.32224040632054174 0.011997742663656885 0.5744469525959367
x =  10
9 -> 0.2926354401805869 0.012042889390519187 0.5744469525959367
x =  11
9 -> 0.2685214446952596 0.01260722347629797 0.5744469525959367
x =  12
9 -> 0.2483803611738149 0.012641083521444697 0.5744469525959367
x =  13
9 -> 0.23149548532731376 0.012127539503386006 0.5744469525959367